# Despliegue con interfaz gráfica – Precio de Airbnb en Nueva York (2019)

Este notebook (punto 3j) prepara y prueba el despliegue del modelo final del notebook **Modelo_Airbnb_NYC** en una aplicación web hecha con **Streamlit**, publicada en **Streamlit Community Cloud** desde el repositorio de GitHub.

👉 **Aplicación publicada:** https://airbnbapp-c8wca8smbq7ohcvhcgitgh.streamlit.app/

**Archivos que usa el despliegue** (todos en el repositorio):

| Archivo | Contenido |
|---|---|
| `app.py` | Aplicación Streamlit |
| `features.py` | Funciones de preparación e ingeniería de características que usa el pipeline |
| `models/modelo_final_airbnb.pkl` | Pipeline final (preparación + características + preprocesamiento + modelo) |
| `models/metricas_modelo.json` | Métricas del modelo final; la app toma de aquí el error típico y el rango de referencia |
| `data/ubicaciones_distritos.csv` | Coordenadas de los alojamientos, para deducir el distrito y el micro-barrio del punto marcado en el mapa |
| `requirements.txt` | Librerías y versiones |

**Pasos**
1. Cargar `features.py` y el modelo final
2. Probar el modelo con alojamientos de ejemplo
3. Probar la aplicación Streamlit de forma automática
4. Fijar las versiones en `requirements.txt`
5. Publicar en Streamlit Community Cloud

*En Colab: subir `features.py`, `modelo_final_airbnb.pkl`, `metricas_modelo.json`, `ubicaciones_distritos.csv` y `app.py`. En el repositorio se ejecuta desde la carpeta `notebooks/`.*

# 1. Cargar `features.py` y el modelo final

El pipeline guardado usa las funciones `preparar_datos` y `crear_variables` de `features.py`. Ese archivo debe estar junto al modelo y ser el mismo con el que se entrenó; por eso **no se reescribe aquí**, sino que se importa el del repositorio.

In [ ]:
import os
import sys
import json
import joblib
import pandas as pd

RAIZ = next(d for d in ['.', '..'] if os.path.exists(os.path.join(d, 'features.py')))
sys.path.insert(0, RAIZ)
import features

def ruta(*partes):
    completa = os.path.join(RAIZ, *partes)
    return completa if os.path.exists(completa) else partes[-1]

print('Variables de entrada que espera el pipeline:')
print(features.COLUMNAS_ENTRADA)

In [ ]:
modelo = joblib.load(ruta('models', 'modelo_final_airbnb.pkl'))
with open(ruta('models', 'metricas_modelo.json'), encoding='utf-8') as f:
    metricas = json.load(f)

ganador = metricas['mejor_modelo']
fila = next(r for r in metricas['test_modelos_ajustados'] if r['modelo'] == ganador['nombre'])
print('Modelo desplegado:', ganador['nombre'])
print('Hiperparámetros:', ganador['parametros'])
print(f"Evaluación en el 30 % de prueba: MAE = {fila['MAE']:.2f} USD | MedAE = {fila['MedAE']:.2f} USD | "
      f"RMSE = {fila['RMSE']:.2f} USD | R² log = {fila['R2_log']:.3f}")
print('Pasos del pipeline:', [nombre for nombre, _ in modelo.regressor_.steps])

# 2. Prueba del modelo con alojamientos de ejemplo

El pipeline recibe los datos tal como los ingresa un usuario en la app: título, distrito, micro-barrio, coordenadas, tipo de alojamiento y datos del anfitrión. Las distancias, las señales del título, la codificación y el escalado ocurren dentro del pipeline.

In [ ]:
ejemplos = pd.DataFrame({
    'name': ['Luxury apartment with skyline view in Midtown', 'Cozy private room near subway', 'Shared room in the Bronx'],
    'neighbourhood_group': ['Manhattan', 'Brooklyn', 'Bronx'],
    'neighbourhood': ['Midtown', 'Bedford-Stuyvesant', 'Fordham'],
    'latitude': [40.7549, 40.6872, 40.8620],
    'longitude': [-73.9840, -73.9418, -73.8900],
    'room_type': ['Entire home/apt', 'Private room', 'Shared room'],
    'minimum_nights': [2, 1, 1],
    'number_of_reviews': [25, 40, 5],
    'reviews_per_month': [1.2, 2.0, 0.3],
    'calculated_host_listings_count': [1, 1, 2],
    'availability_365': [150, 200, 300],
})

q25, q75 = metricas['rango_referencia_razon']['q25'], metricas['rango_referencia_razon']['q75']
ejemplos['precio_estimado_USD'] = modelo.predict(ejemplos).round(0)
ejemplos['rango_USD'] = [f'{p * q25:.0f} – {p * q75:.0f}' for p in ejemplos['precio_estimado_USD']]
ejemplos[['name', 'neighbourhood', 'room_type', 'precio_estimado_USD', 'rango_USD']]

Los resultados son coherentes con el mercado: el apartamento completo en Midtown con palabras de lujo en el título es el más caro, la habitación privada en Brooklyn tiene un precio intermedio y la habitación compartida en el Bronx es la más económica. El rango corresponde a la mitad central de los errores del modelo en el 30 % de prueba.

# 3. Prueba automática de la aplicación Streamlit

La aplicación (`app.py`) tiene tres secciones:

1. **Ubicación:** el usuario hace clic en un mapa interactivo (`folium` + `streamlit-folium`); la app deduce el distrito y el micro-barrio del alojamiento del dataset más cercano al punto marcado.
2. **Anuncio, alojamiento y anfitrión:** título del anuncio, tipo de alojamiento, noches mínimas, disponibilidad, reseñas y número de anuncios del anfitrión.
3. **Estimación:** al presionar **Estimar precio** muestra el precio sugerido, el rango de referencia y una explicación breve de los factores.

El modelo se carga una sola vez con `@st.cache_resource`. La siguiente celda ejecuta la app con el probador oficial de Streamlit (`AppTest`), llena el formulario, presiona el botón y lee el resultado, sin abrir un navegador.

In [ ]:
from streamlit.testing.v1 import AppTest

directorio_original = os.getcwd()
os.chdir(RAIZ)  # la app busca models/ y data/ desde la ra?z del repositorio
try:
    app = AppTest.from_file(os.path.abspath('app.py'), default_timeout=120).run()
    next(c for c in app.text_input if 'T?tulo' in c.label).set_value('Luxury loft with terrace near subway')
    next(s for s in app.selectbox if 'Tipo de alojamiento' in s.label).set_value('Entire home/apt')
    next(b for b in app.button if 'Estimar precio' in b.label).click().run()

    print('Errores de la app:', [e.value for e in app.exception] or 'ninguno')
    print('Ubicaci?n detectada:', app.info[0].value)
    for metrica in app.metric:
        print(f'{metrica.label}: {metrica.value}')
finally:
    os.chdir(directorio_original)

La aplicación se ejecuta sin errores, detecta el distrito y el micro-barrio del punto por defecto (Times Square) y devuelve el precio sugerido y su rango. Para usarla en el computador: `streamlit run app.py` desde la raíz del repositorio.

# 4. Versiones en `requirements.txt`

Streamlit Community Cloud instala las librer?as de `requirements.txt`. Las versiones de **scikit-learn** y **XGBoost** se fijan iguales a las del entrenamiento; con versiones distintas, el `.pkl` podr?a no cargarse. La app funciona con la versi?n de Python ya configurada en el despliegue (3.12 o 3.13); no hace falta cambiarla.

In [ ]:
import sklearn
import xgboost
import streamlit

print('Python:', sys.version.split()[0])
print('scikit-learn:', sklearn.__version__)
print('xgboost:', xgboost.__version__)
print('streamlit:', streamlit.__version__)

requeridas = {'scikit-learn': sklearn.__version__, 'xgboost': xgboost.__version__}
contenido = open(ruta('requirements.txt')).read()
print('\nrequirements.txt actual:\n' + contenido)
for libreria, version in requeridas.items():
    estado = 'coincide' if f'{libreria}=={version}' in contenido else 'NO coincide: actualizar'
    print(f'{libreria}=={version}: {estado}')

# 5. Publicaci?n en Streamlit Community Cloud

1. Subir el repositorio a GitHub con la estructura de la tabla inicial.
2. En https://share.streamlit.io elegir **New app**, el repositorio, la rama `main` y el archivo principal `app.py`.
3. Mantener la versi?n de Python ya configurada (3.12 o 3.13); el `.pkl` carga con las versiones fijadas de scikit-learn y XGBoost, as? que no hace falta redesplegar para cambiarla.
4. Cada vez que se actualiza `models/modelo_final_airbnb.pkl` en GitHub, Streamlit Cloud vuelve a publicar la app autom?ticamente.

La app publicada est? en https://airbnbapp-c8wca8smbq7ohcvhcgitgh.streamlit.app/